# 01 — Download Nutrition5k to Google Drive

Pulls only what the pipeline uses: **overhead RGB-D imagery, metadata, and split ids** (~15–25 GB). The full dataset is 181.4 GB; the difference is side-angle video we never read. License: CC BY 4.0.

Everything lands in the project Drive folder — **no Google Cloud account, project, or gcloud tooling involved**: the files come straight off the public bucket over plain HTTPS.

The copy is **resumable** — rerun after any disconnect and it skips files already in Drive. No GPU needed for this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to the mounted path of the project folder in YOUR Drive.
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

DATA = f'{DRIVE_ROOT}/data/nutrition5k'
!mkdir -p "{DATA}"
print('data root:', DATA)

In [ ]:
# Anonymous HTTPS sync from the public bucket — resumable, Drive-only.
import concurrent.futures
import pathlib
import requests

BUCKET = 'nutrition5k_dataset'
PREFIXES = [
    'nutrition5k_dataset/imagery/realsense_overhead/',
    'nutrition5k_dataset/metadata/',
    'nutrition5k_dataset/dish_ids/',
]
API = f'https://storage.googleapis.com/storage/v1/b/{BUCKET}/o'


def list_objects(prefix):
    token = None
    while True:
        params = {'prefix': prefix, 'fields': 'items(name,size),nextPageToken', 'maxResults': 5000}
        if token:
            params['pageToken'] = token
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        for item in payload.get('items', []):
            yield item['name'], int(item.get('size', 0))
        token = payload.get('nextPageToken')
        if not token:
            break


def fetch(job):
    name, size = job
    dest = pathlib.Path(DATA) / name.removeprefix('nutrition5k_dataset/')
    if dest.exists() and dest.stat().st_size == size:
        return 0  # already synced
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://storage.googleapis.com/{BUCKET}/{name}'
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(dest)
    return 1


jobs = [j for p in PREFIXES for j in list_objects(p)]
print(f'{len(jobs)} objects, {sum(s for _, s in jobs) / 1e9:.1f} GB total')

downloaded = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    for i, n in enumerate(pool.map(fetch, jobs), 1):
        downloaded += n
        if i % 500 == 0:
            print(f'{i}/{len(jobs)} checked, {downloaded} downloaded')
print(f'complete: {downloaded} new files (rest were already in Drive)')

In [ ]:
# Verify: ~5,000 dish folders, each with rgb.png + depth_raw.png, plus metadata CSVs.
!ls "{DATA}/imagery/realsense_overhead" | wc -l
!ls "{DATA}/imagery/realsense_overhead" | head -3
!ls "{DATA}/metadata"
!ls "{DATA}/dish_ids/splits" | head
!du -sh "{DATA}"